# Optional seed-0 training reproducibility

Retrain one EfficientNetV2-S U-Net with the released k07 recipe: all 707 official training observations, seed 0, FOLD=-1, 12,000 updates, BCE plus batch Dice. This is a diagnostic new fit. It does not replace any release file or submit to Kaggle.

Attach only the public snapshot, official competition images and training JSON, and released submitted weights. Training needs ImageNet initialization: enable Internet or mount a compatible pretrained encoder cache as well as the required wheels. Never mount the public MAGFiLO annotation release. No neighbours are required: temporal=False samples annotated frames only. The preparation option skips unused neighbour decoding. FOLD=-1 includes every observation regardless of its nonnegative fold value; the driver creates an all-zero bookkeeping table.

Planning estimate for one T4: roughly 3-5 wall hours for training, map exports and comparisons, expected within 10 GPU hours when one GPU is allocated. This is **an estimate, not a new T4 benchmark**. The archived seed-0 worker printed 7,775 seconds for training plus its map export; shared preparation printed 464 seconds. They came from a concurrent two-device job, not a controlled one-T4 run. Hardware, package versions and download time can change runtime. Training timers and device identity are recorded by this run.

The comparison exports the two released seed maps and replaces only seed 0 with the new fit. It runs the released submitted entry point twice with cached maps and its frozen LightGBM and seed-0 verifier. It reports probability MAE/RMSE, exact pixel fraction, strict mask matches, mean/min matched IoU and a PQ-style agreement **between predictions, not test PQ**. A separate released-map replay versus the archive isolates inference precision differences. No test labels are used.

Outputs include the new checkpoint/maps, probability_agreement.csv, mask_agreement.csv, two prediction CSVs, and training/comparison metadata. Keep the new artifacts separate from the shipped weights. CUDA training is not promised to reproduce exact checkpoint bytes.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CODE = Path("/kaggle/input/REPLACE_CODE_DATASET/filament-segmentation-2026")
INPUT = Path("/kaggle/input/REPLACE_COMPETITION_MOUNT")
WEIGHTS = Path("/kaggle/input/REPLACE_WEIGHT_DATASET")
TRAIN_OUT = Path("/kaggle/working/seed0-training")
COMPARE_OUT = Path("/kaggle/working/seed0-comparison")
WHEELHOUSE = None

assert (CODE / "training/retrain_seed0.py").is_file(), CODE
assert INPUT.is_dir(), INPUT
assert not TRAIN_OUT.exists() and not COMPARE_OUT.exists(), "Choose new output paths."
if WHEELHOUSE is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(WHEELHOUSE), "-r", str(CODE / "requirements.txt")],
                   check=True)
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")


In [ ]:
# A fresh process runs only seed 0; no two-GPU launcher is invoked.
subprocess.run([sys.executable, "-m", "training.retrain_seed0",
                "--input", str(INPUT), "--out", str(TRAIN_OUT)],
               cwd=CODE, env=env, check=True)


In [ ]:
# A new process releases the training optimizer and GPU allocations first.
test_directories = [p for p in INPUT.rglob("test_images") if p.is_dir()]
assert len(test_directories) == 1, test_directories
subprocess.run([
    sys.executable, "-m", "tools.compare_retrained_seed",
    "--images", str(test_directories[0]),
    "--trained-maps", str(TRAIN_OUT / "test_effv2s_s0"),
    "--trained-checkpoint", str(TRAIN_OUT / "effv2s_s0_fold-1.pt"),
    "--weights-dir", str(WEIGHTS), "--out", str(COMPARE_OUT),
    "--device", "cuda:0", "--tta-batch", "4", "--expected-images", "180",
], cwd=CODE, env=env, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(COMPARE_OUT / "probability_agreement.csv").describe())
display(pd.read_csv(COMPARE_OUT / "mask_agreement.csv"))
print((TRAIN_OUT / "training_metadata.json").read_text())
print((COMPARE_OUT / "comparison_metadata.json").read_text())
